# From MR contrast to an inverse problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/guanhuaw/MIRTorch/blob/master/examples/demo_mr_physics.ipynb)

This self-contained tutorial follows one measurement all the way through

$$
\text{tissue parameters}\rightarrow\text{sequence contrast }x
\rightarrow\text{encoding }A\rightarrow\text{noisy data }y
\rightarrow\text{regularized reconstruction }\hat x.
$$

**Learning goals**

- connect $T_1$, $T_2$, proton density, and sequence timing to a complex image;
- turn receive-coil and Fourier encoding into the discrete model $y=Ax+\epsilon$;
- distinguish the adjoint $A^Hy$ from an inverse and expose a sampling null space;
- derive a quadratic reconstruction solved by CG; and
- solve a sparse reconstruction with FISTA and check numerical and image evidence.

The experiment uses a small synthetic phantom and runs without downloads or optional
NUFFT libraries. It is a bridge to reconstruction, not a full Bloch or pulse-sequence
simulator.

In [ ]:
# In Colab, clone MIRTorch and install its example dependencies.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    repository = Path("/content/MIRTorch")
    repository_url = "https://github.com/guanhuaw/MIRTorch.git"
    if not (repository / "pyproject.toml").exists():
        clone = ["git", "clone", "--depth", "1", repository_url, str(repository)]
        subprocess.run(clone, check=True)
    pip = [sys.executable, "-m", "pip", "install", "--quiet", "--editable"]
    subprocess.run([*pip, f"{repository}[examples]"], check=True)
    os.chdir(repository)
    print(f"Colab setup complete: {repository}")

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import torch

from mirtorch.alg import CG, FISTA
from mirtorch.linear import Diffnd, FFTCn, Sense, Wavelet2D
from mirtorch.prox import L1Regularizer
from mirtorch.util import l2_norm


def mps_supports_complex_fft():
    backend = getattr(torch.backends, "mps", None)
    if backend is None or not backend.is_available():
        return False
    try:
        probe = torch.ones(2, dtype=torch.complex64, device="mps")
        torch.fft.fft(probe)
        torch.mps.synchronize()
        return True
    except (RuntimeError, NotImplementedError):
        return False


def best_available_device():
    override = os.environ.get("MIRTORCH_EXAMPLE_DEVICE")
    if override:
        return torch.device(override)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if mps_supports_complex_fft():
        return torch.device("mps")
    return torch.device("cpu")


def magnitude(value):
    return value.detach().abs().cpu().squeeze().numpy()


def complex_normal(shape, *, device):
    real = torch.randn(shape, device=device)
    imag = torch.randn(shape, device=device)
    return torch.complex(real, imag) / math.sqrt(2)


torch.manual_seed(11)
device = best_available_device()
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. From relaxation to image contrast

The Bloch equation models precession and relaxation of the magnetization vector
$\mathbf M=(M_x,M_y,M_z)$:

$$
\frac{d\mathbf M}{dt}
=\gamma\,\mathbf M\times\mathbf B
-\frac{M_x\hat{\mathbf x}+M_y\hat{\mathbf y}}{T_2}
-\frac{M_z-M_0}{T_1}\hat{\mathbf z}.
$$

For a local isochromat after excitation, two useful solutions are

$$
M_z(t)=M_0-(M_0-M_z(0))e^{-t/T_1},\qquad
M_{xy}(t)=M_{xy}(0)e^{-t/T_2}e^{-i2\pi\Delta f t}.
$$

$T_1$ controls longitudinal recovery and $T_2$ controls irreversible transverse
decay. Unresolved static-frequency spread produces an effective
$T_2^*\leq T_2$ in free induction and gradient echoes, whereas an ideal 180-degree
pulse refocuses static offsets at the spin echo. That distinction motivates the
$T_2$ factor in the simple ideal 90-degree spin-echo approximation below:

$$
x(\mathbf r;\mathrm{TR},\mathrm{TE})
=\rho(\mathbf r)\left(1-e^{-\mathrm{TR}/T_1(\mathbf r)}\right)
e^{-\mathrm{TE}/T_2(\mathbf r)}e^{i\phi(\mathbf r)}.
$$

It ignores slice profiles, imperfect refocusing, exchange, diffusion, motion, and
transient states. Its purpose is to show that the reconstructed image $x$ is
sequence-dependent transverse magnetization, not simply proton density $\rho$.
The MRI operator notebook later adds mean B0 evolution during the readout.

In [ ]:
N = 96
axis = torch.linspace(-1, 1, N, device=device)
yy, xx = torch.meshgrid(axis, axis, indexing="ij")


def ellipse(cx, cy, rx, ry, angle=0.0):
    cosine, sine = math.cos(angle), math.sin(angle)
    x_rotated = cosine * (xx - cx) + sine * (yy - cy)
    y_rotated = -sine * (xx - cx) + cosine * (yy - cy)
    return x_rotated.square() / rx**2 + y_rotated.square() / ry**2 <= 1


brain = ellipse(0.0, 0.0, 0.72, 0.90)
white_matter = ellipse(0.0, 0.00, 0.54, 0.70)
ventricles = ellipse(-0.13, 0.02, 0.075, 0.20, 0.12)
ventricles |= ellipse(0.13, 0.02, 0.075, 0.20, -0.12)

tissue = torch.zeros((N, N), dtype=torch.long, device=device)
tissue[brain] = 1  # gray matter
tissue[white_matter] = 2
tissue[ventricles] = 3  # CSF

rho = torch.zeros_like(xx)
t1_ms = torch.ones_like(xx)
t2_ms = torch.ones_like(xx)
properties = {
    1: (0.86, 1350.0, 100.0),
    2: (0.70, 850.0, 70.0),
    3: (1.00, 4000.0, 2000.0),
}
for label, (density, t1_value, t2_value) in properties.items():
    region = tissue == label
    rho[region] = density
    t1_ms[region] = t1_value
    t2_ms[region] = t2_value

# Smooth object phase makes x genuinely complex; coil phase remains in s_c.
object_phase = brain * (0.35 * xx - 0.20 * yy)


def spin_echo(tr_ms, te_ms):
    amplitude = rho * (1 - torch.exp(-tr_ms / t1_ms))
    amplitude *= torch.exp(-te_ms / t2_ms)
    return amplitude * torch.exp(1j * object_phase) * brain


contrasts = {
    "PD-like\nTR 8000, TE 10 ms": spin_echo(8000.0, 10.0),
    "T1-weighted\nTR 500, TE 15 ms": spin_echo(500.0, 15.0),
    "T2-weighted\nTR 3000, TE 100 ms": spin_echo(3000.0, 100.0),
}

wm = tissue == 2
gm = tissue == 1
csf = tissue == 3
t1_weighted = contrasts["T1-weighted\nTR 500, TE 15 ms"].abs()
t2_weighted = contrasts["T2-weighted\nTR 3000, TE 100 ms"].abs()
assert t1_weighted[wm].mean() > t1_weighted[gm].mean() > t1_weighted[csf].mean()
assert t2_weighted[csf].mean() > t2_weighted[gm].mean() > t2_weighted[wm].mean()

chosen_signal = contrasts["T1-weighted\nTR 500, TE 15 ms"]
x_true = (chosen_signal / chosen_signal.abs().amax())[None, None].to(torch.complex64)

fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
parameter_maps = (
    (rho, "Proton density"),
    (t1_ms * brain, "$T_1$ (ms)"),
    (t2_ms * brain, "$T_2$ (ms)"),
)
for current_axis, (image, title) in zip(axes[0], parameter_maps):
    artist = current_axis.imshow(image.detach().cpu(), cmap="viridis")
    current_axis.set_title(title)
    fig.colorbar(artist, ax=current_axis, shrink=0.72)
for current_axis, (title, image) in zip(axes[1], contrasts.items()):
    current_axis.imshow(image.abs().detach().cpu(), cmap="gray")
    current_axis.set_title(title)
for current_axis in axes.flat:
    current_axis.axis("off")
plt.show()

## 2. Gradient and coil encoding

A gradient waveform accumulates spatial phase. With
$\bar\gamma=\gamma/(2\pi)$ in Hz/T,

$$
\mathbf k(t)=\bar\gamma\int_0^t\mathbf G(\tau)\,d\tau,
$$

and receive coil $c$ measures approximately

$$
y_c(t_j)=\int_\Omega s_c(\mathbf r)x(\mathbf r)
e^{-i2\pi\mathbf k(t_j)\cdot\mathbf r}\,d\mathbf r+\epsilon_c(t_j).
$$

After voxel discretization, Cartesian multi-coil sampling becomes
$y=MFSx+\epsilon$. $S$ applies sensitivity maps, $F$ is a centered Fourier
transform, and $M$ retains acquired phase-encoding lines. `Sense` implements this
composition without storing a dense matrix. MIRTorch's centered
`FFTCn(norm="ortho")` is a unitary discrete convention, so voxel-volume and
receiver-gain constants are absorbed into the chosen data normalization.
Measured data and simulations must therefore use a consistent normalization.

### One gradient moment is one Fourier sample

Before adding coils, consider a one-dimensional object at normalized positions
$r_n$. A gradient moment selects spatial frequency $k$, and the receiver sums
the resulting phasors:

$$
y(k)=\frac{1}{\sqrt N}\sum_n x_n e^{-i2\pi k r_n}.
$$

The direct phase sum below agrees with a centered orthonormal FFT. The middle
panel shows the real part of one encoding phasor; changing the gradient moment
changes its spatial winding rate.

In [ ]:
line_length = 64
position = (
    torch.arange(line_length, dtype=torch.float32, device=device) - line_length // 2
) / line_length
line_object = (
    ((position > -0.32) & (position < -0.12)).float()
    + 0.65 * ((position > 0.08) & (position < 0.27)).float()
).to(torch.complex64)
frequency = torch.arange(
    -line_length // 2,
    line_length // 2,
    dtype=torch.float32,
    device=device,
)
encoding_phase = torch.exp(-2j * math.pi * frequency[:, None] * position[None])
direct_samples = encoding_phase @ line_object / math.sqrt(line_length)
fft_samples = torch.fft.fftshift(
    torch.fft.fft(torch.fft.ifftshift(line_object), norm="ortho")
)
direct_fft_error = (l2_norm(direct_samples - fft_samples) / l2_norm(fft_samples)).item()
print(f"Direct phase sum vs centered FFT error: {direct_fft_error:.2e}")
assert direct_fft_error < 5e-5

selected_frequency = 8
selected_index = line_length // 2 + selected_frequency
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), constrained_layout=True)
axes[0].plot(position.cpu(), line_object.real.cpu())
axes[0].set(title="1D transverse magnetization", xlabel="Position / FOV")
axes[1].plot(position.cpu(), encoding_phase[selected_index].real.cpu())
axes[1].set(title=f"Encoding phasor, k={selected_frequency}", xlabel="Position / FOV")
axes[2].stem(frequency.cpu(), direct_samples.abs().cpu(), basefmt=" ")
axes[2].set(title="Magnitude of Fourier samples", xlabel="Cycles / FOV")
for current_axis in axes:
    current_axis.grid(alpha=0.2)
plt.show()

### Add receiver coils and undersampling

We now move from the one-dimensional Fourier identity to a two-dimensional,
four-coil acquisition. Smooth complex coil maps are RSS-normalized, and a
Cartesian mask retains every fourth phase-encoding line plus the center.

In [ ]:
coil_maps = []
for angle in torch.linspace(0, 2 * math.pi, 5, device=device)[:-1]:
    center_x = 1.10 * torch.cos(angle)
    center_y = 1.10 * torch.sin(angle)
    coil_magnitude = (
        torch.exp(
            -((xx - center_x).square() + (yy - center_y).square()) / (2 * 0.55**2)
        )
        + 0.04
    )
    coil_phase = 0.9 * (xx * torch.sin(angle) - yy * torch.cos(angle))
    coil_maps.append(coil_magnitude * torch.exp(1j * coil_phase))

sensitivity_maps = torch.stack(coil_maps).to(torch.complex64)
sensitivity_maps /= torch.sqrt(sensitivity_maps.abs().square().sum(0, keepdim=True))
sensitivity_maps = sensitivity_maps[None]

sampling_mask = torch.zeros((N, N), device=device)
sampling_mask[::4] = 1
sampling_mask[N // 2 - 4 : N // 2 + 4] = 1
sampling_mask = sampling_mask[None]

A_full = Sense(sensitivity_maps, torch.ones_like(sampling_mask))
A = Sense(sensitivity_maps, sampling_mask)
y_clean = A * x_true

coil_power = sensitivity_maps.abs().square().sum(1, keepdim=True)
rss_error = (coil_power - 1).abs().amax().item()
full_encoding_error = (
    l2_norm(A_full.H * (A_full * x_true) - x_true) / l2_norm(x_true)
).item()
sample_fraction = sampling_mask.mean().item()
print(
    f"Acquired {sample_fraction:.1%} of k-space (effective R={1 / sample_fraction:.2f})"
)
print(f"RSS coil normalization error: {rss_error:.2e}")
print(f"Fully sampled A^H A identity error: {full_encoding_error:.2e}")
assert rss_error < 5e-5
assert full_encoding_error < 5e-4

### Numerical checks are part of the model

For complex data, the adjoint must satisfy
$\langle Ax,z\rangle=\langle x,A^Hz\rangle$. This is stronger than checking
array shapes. RSS-normalized coils and a unitary FFT also imply that the fully
sampled operator obeys $A^HAx=x$ in this synthetic experiment.

We then add white complex Gaussian noise at a known norm SNR. Real receiver noise
can be correlated across coils; prewhitening is needed before an unweighted
least-squares model is appropriate.

In [ ]:
probe_image = complex_normal(A.size_in, device=device)
probe_data = complex_normal(A.size_out, device=device)
left_inner_product = ((A * probe_image).conj() * probe_data).sum()
right_inner_product = (probe_image.conj() * (A.H * probe_data)).sum()
adjoint_error = (
    (left_inner_product - right_inner_product).abs()
    / torch.maximum(left_inner_product.abs(), right_inner_product.abs()).clamp_min(1e-8)
).item()

target_snr_db = 32.0
noise = complex_normal(y_clean.shape, device=device) * sampling_mask[:, None]
target_noise_norm = l2_norm(y_clean) / (10 ** (target_snr_db / 20))
noise = noise * (target_noise_norm / l2_norm(noise))
measurements = y_clean + noise
measured_snr_db = 20 * torch.log10(l2_norm(y_clean) / l2_norm(noise)).item()
x_adjoint = A.H * measurements

print(f"Complex adjoint relative error: {adjoint_error:.2e}")
print(f"Measurement SNR: {measured_snr_db:.2f} dB")
assert adjoint_error < 5e-4
assert abs(measured_snr_db - target_snr_db) < 0.05

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), constrained_layout=True)
axes[0].imshow(sensitivity_maps[0, 0].abs().cpu(), cmap="viridis")
axes[0].set_title("Coil 1 magnitude")
axes[1].imshow(sampling_mask[0].cpu(), cmap="gray", aspect="auto")
axes[1].set_title("Sampling mask")
log_kspace = torch.log1p(y_clean[0].abs().square().sum(0).sqrt())
axes[2].imshow(log_kspace.cpu(), cmap="magma")
axes[2].set_title("Log RSS k-space")
axes[3].imshow(magnitude(x_adjoint), cmap="gray", vmin=0, vmax=1)
axes[3].set_title("Adjoint $A^Hy$")
for current_axis in axes:
    current_axis.axis("off")
plt.show()

## 3. Why $A^Hy$ is not an inverse

$A^H$ reverses the data flow, but undersampling discards information. To make
nonuniqueness explicit, first ignore coil encoding and put the omitted Fourier
coefficients of $x$ into an image $h$. Then $A_1h\approx0$, so $x$ and $x+h$
produce the same acquired single-coil data even though they are visibly different.

Coil sensitivities provide additional encoding and reduce this ambiguity, but
undersampling, noise, conditioning, and model mismatch still make reconstruction
an inverse problem.

In [ ]:
fourier = FFTCn(x_true.shape, x_true.shape, dims=(2, 3), norm="ortho")
missing_samples = (1 - sampling_mask)[:, None]
null_perturbation = fourier.H * (missing_samples * (fourier * x_true))
single_coil = Sense(
    torch.ones((1, 1, N, N), dtype=torch.complex64, device=device),
    sampling_mask,
)
null_measurement_error = (
    l2_norm(single_coil * null_perturbation) / l2_norm(single_coil * x_true)
).item()
relative_perturbation = (l2_norm(null_perturbation) / l2_norm(x_true)).item()

print(f"Relative measurement from h: {null_measurement_error:.2e}")
print(f"Relative image change ||h||/||x||: {relative_perturbation:.3f}")
assert null_measurement_error < 5e-4
assert relative_perturbation > 0.10

fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
null_images = (
    (x_true, "$x$"),
    (x_true + null_perturbation, "$x+h$"),
    (null_perturbation, "$h$ in omitted frequencies"),
)
for current_axis, (image, title) in zip(axes, null_images):
    current_axis.imshow(magnitude(image), cmap="gray", vmin=0, vmax=1)
    current_axis.set_title(title)
    current_axis.axis("off")
plt.show()

## 4. Gaussian likelihood plus a quadratic image model

White complex Gaussian noise gives the data term
$\tfrac12\|Ax-y\|_2^2$. A quadratic roughness penalty leads to

$$
\hat x_{\mathrm{Q}}=\arg\min_x
\frac12\|Ax-y\|_2^2+\frac\beta2\|Dx\|_2^2,
$$

whose stationary point solves

$$
(A^HA+\beta D^HD)\hat x_{\mathrm{Q}}=A^Hy.
$$

The system is Hermitian positive definite on this problem, so conjugate gradients
needs only operator applications. The illustrative $\beta$ trades data fit for
noise and alias suppression; it is not a universal setting.

In [ ]:
finite_difference = Diffnd(x_true.shape, dims=(2, 3))
beta = 2e-3
right_hand_side = A.H * measurements
normal_operator = A.H * A + beta * (finite_difference.H * finite_difference)
right_hand_side_norm = l2_norm(right_hand_side)


def relative_system_residual(residual):
    return (l2_norm(residual) / right_hand_side_norm).item()


cg_result = CG(
    normal_operator,
    max_iter=35,
    tol=0,
    eval_func=relative_system_residual,
).run(torch.zeros_like(x_true), right_hand_side, return_info=True)
x_cg = cg_result.solution
print(
    f"CG: {cg_result.iterations} iterations, "
    f"final relative system residual {cg_result.history[-1]:.2e}"
)
assert cg_result.history[-1] < 1e-3

## 5. Change the image model: wavelet sparsity

Edges are not naturally Gaussian. An orthonormal wavelet model instead solves

$$
\hat x_{\mathrm{W}}=\arg\min_x
\frac12\|Ax-y\|_2^2+\lambda\|Wx\|_1.
$$

For a regularizer $R$, its proximal map is

$$
\operatorname{prox}_{\alpha R}(v)
=\arg\min_u\frac12\|u-v\|_2^2+\alpha R(u).
$$

Soft thresholding is the proximal map of an $\ell_1$ penalty. FISTA alternates
a data-consistency gradient with this proximal operation. Because the coils are
RSS-normalized, the mask and orthonormal FFT are contractions, so
$\|A\|_2^2\leq1$. We use a small numerical margin for the Lipschitz bound; this
bound controls a safe step size and is distinct from choosing $\lambda$.

This section changes the objective as well as the algorithm, so its results compare
complete reconstruction models, not FISTA against CG on one objective. For controlled
comparisons of several solvers on the same objective, continue with
[`demo_cs.ipynb`](https://colab.research.google.com/github/guanhuaw/MIRTorch/blob/master/examples/demo_cs.ipynb).

In [ ]:
wavelet = Wavelet2D(
    x_true.shape,
    wave_type="db4",
    padding="periodization",
    J=3,
    device=device,
)
wavelet_weight = 2e-3
data_lipschitz = 1.001 * coil_power.amax().item()
wavelet_prox = L1Regularizer(wavelet_weight, T=wavelet)


def data_gradient(value):
    return A.H * (A * value - measurements)


def wavelet_objective(value):
    data_term = 0.5 * (A * value - measurements).abs().square().sum()
    regularizer = wavelet_weight * (wavelet * value).abs().sum()
    return (data_term + regularizer).item()


initial_wavelet_objective = wavelet_objective(x_adjoint)
fista_result = FISTA(
    data_gradient,
    data_lipschitz,
    wavelet_prox,
    max_iter=80,
    restart=True,
    eval_func=wavelet_objective,
).run(x_adjoint, return_info=True)
x_fista = fista_result.solution
objective_ratio = fista_result.history[-1] / initial_wavelet_objective
print(
    f"FISTA: {fista_result.iterations} iterations, "
    f"objective ratio {objective_ratio:.3f}"
)
assert objective_ratio < 0.60

## 6. Compare data fit, image error, and convergence

No single diagnostic is enough. The normalized data residual checks acquired
samples, while NRMSE and PSNR are available only because this is a simulation.
The same display scale and absolute-error scale make visual comparisons honest.

The quadratic and wavelet estimates use different priors and regularization weights; their metrics are evidence about the complete reconstruction choices, not a general ranking of CG and FISTA.

In [ ]:
def reconstruction_metrics(value):
    magnitude_error = value.abs() - x_true.abs()
    nrmse = (l2_norm(magnitude_error) / l2_norm(x_true.abs())).item()
    rmse = magnitude_error.square().mean().sqrt()
    psnr = (20 * torch.log10(x_true.abs().amax() / rmse)).item()
    data_residual = (l2_norm(A * value - measurements) / l2_norm(measurements)).item()
    return nrmse, psnr, data_residual


reconstructions = {
    "Adjoint": x_adjoint,
    "Quadratic CG": x_cg,
    "Wavelet FISTA": x_fista,
}
metric_table = {
    name: reconstruction_metrics(image) for name, image in reconstructions.items()
}

print(f"{'method':<16} {'NRMSE':>9} {'PSNR (dB)':>12} {'data residual':>15}")
for name, (nrmse, psnr, residual) in metric_table.items():
    print(f"{name:<16} {nrmse:9.4f} {psnr:12.2f} {residual:15.4f}")

adjoint_nrmse, adjoint_psnr, adjoint_residual = metric_table["Adjoint"]
cg_nrmse, _, _ = metric_table["Quadratic CG"]
fista_nrmse, fista_psnr, fista_residual = metric_table["Wavelet FISTA"]
assert torch.isfinite(x_cg).all() and torch.isfinite(x_fista).all()
assert cg_nrmse < 0.75 * adjoint_nrmse
assert fista_nrmse < 0.75 * adjoint_nrmse
assert fista_psnr > adjoint_psnr + 5
assert fista_residual < 0.60 * adjoint_residual

reference = magnitude(x_true)
error_maps = {
    name: abs(magnitude(image) - reference) for name, image in reconstructions.items()
}
error_max = float(torch.quantile(torch.as_tensor(error_maps["Adjoint"]), 0.995))
fig, axes = plt.subplots(2, 4, figsize=(13, 6.2), constrained_layout=True)
axes[0, 0].imshow(reference, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("Reference")
axes[1, 0].axis("off")
for column, (name, image) in enumerate(reconstructions.items(), start=1):
    nrmse, psnr, _ = metric_table[name]
    axes[0, column].imshow(magnitude(image), cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(f"{name}\nNRMSE {nrmse:.3f}, {psnr:.1f} dB")
    error_artist = axes[1, column].imshow(
        error_maps[name], cmap="magma", vmin=0, vmax=error_max
    )
    axes[1, column].set_title("Absolute magnitude error")
for current_axis in axes.flat:
    current_axis.axis("off")
fig.colorbar(error_artist, ax=axes[1, 1:].tolist(), shrink=0.75)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
axes[0].semilogy(cg_result.history)
axes[0].set(
    title="CG normal-system convergence",
    xlabel="Iteration",
    ylabel="Relative residual",
)
fista_curve = [initial_wavelet_objective, *fista_result.history]
axes[1].semilogy(
    torch.as_tensor(fista_curve) / initial_wavelet_objective,
    color="tab:orange",
)
axes[1].set(
    title="FISTA wavelet objective",
    xlabel="Iteration",
    ylabel="Objective / initial objective",
)
for current_axis in axes:
    current_axis.grid(alpha=0.25)
plt.show()

## Takeaway and next steps

This experiment separates three questions that are easy to conflate:

1. **Physics:** what complex transverse magnetization did the sequence create?
2. **Encoding:** how did coils and gradients turn that image into samples?
3. **Inference:** what additional image model makes incomplete noisy data useful?

The adjoint test validates code, the null-space example diagnoses nonuniqueness, CG
solves a quadratic model, and FISTA uses a nonsmooth sparse prior. Good convergence
does not guarantee good physics: B0, motion, trajectory errors, and sensitivity-map
errors must still be modeled or checked.

Continue with:

- [`demo_mri.ipynb`](https://colab.research.google.com/github/guanhuaw/MIRTorch/blob/master/examples/demo_mri.ipynb) for Cartesian/non-Cartesian operators,
  CG-SENSE, and B0 correction;
- [`demo_cs.ipynb`](https://colab.research.google.com/github/guanhuaw/MIRTorch/blob/master/examples/demo_cs.ipynb) for wavelet/TV priors and
  FISTA/POGM/FBPD comparisons; and
- [`demo_trajectory_optimization.ipynb`](https://colab.research.google.com/github/guanhuaw/MIRTorch/blob/master/examples/demo_trajectory_optimization.ipynb)
  for differentiation through acquisition and reconstruction.

**Background:** Fessler's chapters on
[Fourier measurements](https://web.eecs.umich.edu/~fessler/book/c-four.pdf) and
[MR reconstruction](https://web.eecs.umich.edu/~fessler/book/c-mr.pdf), and
[Bloch's nuclear-induction model](https://doi.org/10.1103/PhysRev.70.460).